# Titanic Survival Prediction — Scikit-learn Pipeline

This notebook builds a reproducible classification workflow using `ColumnTransformer` and `Pipeline`. It follows the key learning ideas from the CampusX pipelines lesson, with a cleaner column-based preprocessing setup and APIs compatible with current scikit-learn versions.

**Workflow:** load data → split data → preprocess numeric/categorical columns → train a model → evaluate → cross-validate → tune hyperparameters → save the fitted model.

## 1. Imports

We use a fixed random seed so the split and decision tree are reproducible.

In [ ]:
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

## 2. Load and inspect the dataset

Keep `train.csv` in the same directory as this notebook when running it from the repository root.

In [ ]:
df = pd.read_csv("train.csv")
display(df.head())
print(f"Rows: {df.shape[0]}, columns: {df.shape[1]}")
display(df.isna().sum().sort_values(ascending=False))

## 3. Define features and target

`Survived` is the target. We exclude identifiers and high-cardinality text fields for this introductory model. Unlike changing the original dataframe in place, creating `X` and `y` separately keeps the source data intact.

In [ ]:
target = "Survived"
drop_columns = ["Survived", "PassengerId", "Name", "Ticket", "Cabin"]

X = df.drop(columns=drop_columns, errors="ignore")
y = df[target]

numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 4. Split into training and test sets

We split before fitting any preprocessing steps. `stratify=y` keeps the survival-class proportions approximately similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## 5. Build preprocessing transformers

- Numeric columns: fill missing values with the median, then scale.
- Categorical columns: fill missing values with the most frequent category, then one-hot encode.

`handle_unknown="ignore"` lets the encoder handle categories that appear at prediction time but were not seen during fitting. The transformations are column-based, so they don't depend on fragile numeric column positions.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

## 6. Create and train the full pipeline

The final estimator is a decision tree. Putting preprocessing and the model in one pipeline makes cross-validation safer because preprocessing is fitted only on each training fold.

In [ ]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
])

pipeline.fit(X_train, y_train)

## 7. Evaluate on the held-out test set

Accuracy is a useful starting point, but read precision, recall, and F1-score as well. Do not compare results from different splits as if they were directly equivalent.

In [ ]:
y_pred = pipeline.predict(X_test)
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.3f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)

## 8. Cross-validation

Cross-validation estimates how consistently the pipeline performs across training folds. The held-out test set remains separate for final evaluation.

In [ ]:
cv_scores = cross_val_score(
    pipeline, X_train, y_train, cv=5, scoring="accuracy"
)
print("Fold accuracies:", cv_scores.round(3))
print(f"Mean CV accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

## 9. Tune hyperparameters with GridSearchCV

The double underscore (`model__`) accesses parameters of the pipeline's final step. Grid search runs cross-validation using only the training split.

In [ ]:
param_grid = {
    "model__max_depth": [2, 3, 4, 5, 8, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    refit=True,
)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print(f"Best mean CV accuracy: {grid_search.best_score_:.3f}")

## 10. Evaluate the tuned pipeline

This is the final evaluation of the model selected by cross-validation. Treat this test score as an estimate for this split, not a guarantee of real-world performance.

In [ ]:
best_pipeline = grid_search.best_estimator_
tuned_predictions = best_pipeline.predict(X_test)

print(f"Tuned test accuracy: {accuracy_score(y_test, tuned_predictions):.3f}")
print(classification_report(y_test, tuned_predictions, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, tuned_predictions)

## 11. Save the fitted pipeline

We save the **best fitted pipeline** so inference uses the exact same preprocessing and model. Only load model files you trust, because joblib/pickle-based formats can execute code when loaded.

In [ ]:
joblib.dump(best_pipeline, "titanic_pipeline.joblib")
print("Saved fitted pipeline to titanic_pipeline.joblib")

## Key takeaways

- Fit preprocessing on training data only; avoid data leakage.
- Use `ColumnTransformer` to apply different preparation steps to different column types.
- Use `Pipeline` to keep preprocessing and modelling together.
- Use cross-validation and grid search for model selection, then evaluate once on held-out test data.
- Rerun the notebook locally to generate fresh outputs for your environment.